# CloudSEN12 tile-discard analysis (reference labels) — DPR evidence
Run cells **in order**. Each cell prints a check; if a check fails (`AssertionError`), **stop and send the output**.

Outputs (in `/content/out/`): `tiles.csv` (golden decisions), `labels/tile_XXX.hex` (RTL testbench input),
`summary.txt`, `hist.png`, `demo_tiles.png`, `used_ids.csv`.

In [ ]:
# Cell 0 — install (pinned to the version used on the dataset page)
!pip install -q tacoreader==0.5.3 rasterio

In [ ]:
# Cell 1 — load metadata only (small, fast) and LOOK at it
import tacoreader, rasterio as rio, numpy as np, pandas as pd, os, math
print("tacoreader", tacoreader.__version__)
l1c = tacoreader.load("tacofoundation:cloudsen12-l1c")
ext = tacoreader.load("tacofoundation:cloudsen12-extra")
for name, ds in [("l1c", l1c), ("extra", ext)]:
    print(f"\n===== {name}: {len(ds)} samples =====")
    print(list(ds.columns))
    print(pd.DataFrame(ds.head(2)).T.to_string()[:2500])

In [ ]:
# Cell 2 — find label_type / split / id columns automatically (no guessing)
def cols_containing_value(ds, value):
    out = []
    for c in ds.columns:
        try:
            if ds[c].astype(str).eq(value).any():
                out.append(c)
        except Exception:
            pass
    return out

label_cols = cols_containing_value(l1c, "high")
split_cols = cols_containing_value(l1c, "test")
print("columns containing 'high':", label_cols)
print("columns containing 'test':", split_cols)
assert len(label_cols) == 1, "Need exactly one label_type column -> send me this output"
LABEL_COL = label_cols[0]
SPLIT_COL = split_cols[0] if len(split_cols) == 1 else None
ID_COL = "tortilla:id"
assert ID_COL in l1c.columns and ID_COL in ext.columns, "No tortilla:id column -> send me Cell 1 output"

print(l1c[LABEL_COL].value_counts())
if SPLIT_COL: print(l1c[SPLIT_COL].value_counts())

# Match l1c <-> extra by ID, never by position
ext_pos = {sid: i for i, sid in enumerate(ext[ID_COL].values)}
common = sum(sid in ext_pos for sid in l1c[ID_COL].values)
print(f"l1c samples whose ID exists in extra: {common} / {len(l1c)}")
assert common > 0, "IDs do not match between l1c and extra -> send me this output"

In [ ]:
# Cell 3 — inspect ONE high-quality sample: label + lc10 shapes and values
def open_label(sample_df):
    # dataset page: .read(0) = image, .read(1) = label
    return sample_df.read(1)

def read_lc10(ext_sample_df):
    items = list(ext_sample_df[ID_COL].values)
    # Case A: lc10 is its own item
    for i, it in enumerate(items):
        if str(it).lower() == "lc10":
            with rio.open(ext_sample_df.read(i)) as s:
                return s.read(1)
    # Case B: lc10 is a named band inside one of the items
    for i, it in enumerate(items):
        with rio.open(ext_sample_df.read(i)) as s:
            desc = [str(d).lower() for d in s.descriptions]
            if "lc10" in desc:
                return s.read(desc.index("lc10") + 1)
    raise RuntimeError(f"lc10 not found. extra items = {items}")

hq_pos = np.where(l1c[LABEL_COL].astype(str).values == "high")[0]
p = int(hq_pos[0])
sid = l1c[ID_COL].values[p]
s = l1c.read(p)
print("l1c items in this sample:", list(s[ID_COL].values))
with rio.open(s.read(0)) as src:
    print("image bands:", src.count, "shape:", src.shape, "descriptions:", src.descriptions)
with rio.open(open_label(s)) as src:
    lab = src.read(1)
print("label shape:", lab.shape, "unique:", np.unique(lab, return_counts=True))

e = ext.read(ext_pos[sid])
print("extra items in this sample:", list(e[ID_COL].values))
lc = read_lc10(e)
print("lc10 shape:", lc.shape, "unique:", np.unique(lc))

assert lab.shape == (512, 512), "Label is not 512x512 -> send me this output"
assert lc.shape == lab.shape, "lc10 grid != label grid -> send me this output"
assert set(np.unique(lab)).issubset({0, 1, 2, 3, 99}), "Unexpected label codes -> send me this output"
print("Cell 3 checks passed")

In [ ]:
# Cell 4 — select 50 high-quality patches (test split if available, else seed 42)
N_PATCH = 50
cand = hq_pos
if SPLIT_COL is not None:
    test_mask = l1c[SPLIT_COL].astype(str).values[hq_pos] == "test"
    if test_mask.sum() >= N_PATCH:
        cand = hq_pos[test_mask]
rng = np.random.default_rng(42)
chosen = np.sort(rng.choice(cand, size=N_PATCH, replace=False))
chosen = [int(c) for c in chosen if l1c[ID_COL].values[c] in ext_pos]
print("selected:", len(chosen), "| from test split:", SPLIT_COL is not None and len(cand) < len(hq_pos))
os.makedirs("/content/out/labels", exist_ok=True)
pd.DataFrame({"l1c_position": chosen, "id": [l1c[ID_COL].values[c] for c in chosen]}).to_csv("/content/out/used_ids.csv", index=False)

In [ ]:
# Cell 5 — 3-class map, 256x256 tiles, golden decisions, hex export for the RTL testbench
# Codes (2 bits): 0 = land, 1 = water, 2 = cloud
TILE = 256
NPIX = TILE * TILE                       # 65,536
N_T  = math.ceil(0.70 * NPIX)            # 45,876  (0.7*65536 = 45875.2 -> ceil)
assert N_T == 45876

rows = []
tile_no = 0
for c in chosen:
    sid = l1c[ID_COL].values[c]
    with rio.open(open_label(l1c.read(c))) as src:
        lab = src.read(1)
    lc = read_lc10(ext.read(ext_pos[sid]))

    cls = np.zeros(lab.shape, dtype=np.uint8)            # land by default (incl. shadow)
    water = (lc == 80)                                   # WorldCover 80 = permanent water bodies
    cloud = np.isin(lab, [1, 2])                         # thick + thin cloud
    cls[water] = 1
    cls[cloud] = 2                                       # cloud overrides water
    nodata = int((lab == 99).sum())                      # counted as land, reported separately

    for r in range(2):
        for q in range(2):
            t = cls[r*TILE:(r+1)*TILE, q*TILE:(q+1)*TILE]
            cc = int((t == 2).sum()); wc = int((t == 1).sum())
            np.save(f"/content/out/labels/tile_{tile_no:03d}.npy", t)
            # one hex digit per line, raster order (row by row) -> $readmemh
            with open(f"/content/out/labels/tile_{tile_no:03d}.hex", "w") as f:
                f.write("\n".join(format(v, "x") for v in t.ravel()))
            rows.append(dict(tile=tile_no, id=sid, quadrant=f"r{r}c{q}",
                             cloud_cnt=cc, water_cnt=wc, land_cnt=NPIX-cc-wc,
                             nodata_in_patch=nodata,
                             discard_cloud_only=int(cc >= N_T),
                             discard_cloud_water=int(cc + wc >= N_T)))
            tile_no += 1
    print(f"done patch {len(rows)//4}/{len(chosen)}", end="\r")

df = pd.DataFrame(rows)
df.to_csv("/content/out/tiles.csv", index=False)
# golden file for the testbench: one decision bit per tile, same order as tile_XXX.hex
with open("/content/out/golden_decisions.txt", "w") as f:
    f.write("\n".join(str(v) for v in df.discard_cloud_water))
print("\ntiles:", len(df)); print(df.head())

In [ ]:
# Cell 6 — DPR numbers: discard %, threshold sensitivity, histogram
import matplotlib.pyplot as plt
frac = (df.cloud_cnt + df.water_cnt) / NPIX
lines = [f"Tiles analysed: {len(df)} (from {len(chosen)} CloudSEN12 high-quality patches)",
         f"Pixels: {len(df)*NPIX:,}",
         f"Discard, cloud-only rule (>=70%):    {100*df.discard_cloud_only.mean():.1f} %",
         f"Discard, cloud+water rule (>=70%):   {100*df.discard_cloud_water.mean():.1f} %"]
for th in (0.6, 0.7, 0.8):
    lines.append(f"cloud+water discard at {int(th*100)}%: {100*(frac >= th).mean():.1f} %")
txt = "\n".join(lines); print(txt)
open("/content/out/summary.txt", "w").write(txt)

plt.figure(figsize=(6,4))
plt.hist(frac*100, bins=20, edgecolor="k")
plt.axvline(70, color="r", ls="--", label="70% threshold")
plt.xlabel("cloud + water fraction per 256x256 tile (%)"); plt.ylabel("tiles")
plt.title("CloudSEN12 reference labels"); plt.legend(); plt.tight_layout()
plt.savefig("/content/out/hist.png", dpi=150); plt.show()

In [ ]:
# Cell 7 — demo figure: one cloud-, one water-, one land-dominated tile (RGB + class map)
from matplotlib.colors import ListedColormap
picks = {"cloud": df.cloud_cnt.idxmax(), "water": df.water_cnt.idxmax(), "land": df.land_cnt.idxmax()}
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for k, (name, i) in enumerate(picks.items()):
    row = df.loc[i]; c = chosen[int(row.tile) // 4]
    r, q = int(row.quadrant[1]), int(row.quadrant[3])
    with rio.open(l1c.read(c).read(0)) as src:
        rgb = src.read([4, 3, 2], window=rio.windows.Window(q*TILE, r*TILE, TILE, TILE))
    ax[0, k].imshow(np.clip(rgb.transpose(1, 2, 0) / 3000, 0, 1))
    ax[0, k].set_title(f"tile {int(row.tile)} ({name}) RGB")
    ax[1, k].imshow(np.load(f"/content/out/labels/tile_{int(row.tile):03d}.npy"),
                    cmap=ListedColormap(["tan", "royalblue", "white"]), vmin=0, vmax=2)
    dec = "DISCARD" if row.discard_cloud_water else "KEEP"
    ax[1, k].set_title(f"cloud {row.cloud_cnt} + water {row.water_cnt} -> {dec}")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig("/content/out/demo_tiles.png", dpi=150); plt.show()

In [ ]:
# Cell 8 — zip results to download
!cd /content && zip -qr out.zip out && ls -lh /content/out.zip